# 🎬 VideoStudio — Colab GPU Worker (Wan2.1-T2V-1.3B)

This notebook turns a Google Colab GPU into the video engine for your local **VideoStudio** (`app.py`).
It loads **Wan2.1-T2V-1.3B** and starts a small Gradio server with a public `*.gradio.live` URL.

### Steps
1. **Runtime → Change runtime type** → pick a GPU → **Save**.
   - **Colab Pro:** choose **L4** or **A100** if offered (much faster). With **T4**, also switch **High-RAM** on.
   - **Free Colab (T4, standard RAM):** the 11 GB text encoder does not fit in memory; cell 1 tells you.
2. **Runtime → Run all**. The first run downloads about 17 GB of weights (a few minutes).
3. Copy the `https://xxxxx.gradio.live` URL printed by the **last** cell.
4. In VideoStudio: **⚙️ Settings → More keys (optional) → Colab worker URL** → paste → **💾 Save settings**,
   then press **🔄 Check connection**. It should say *ready (connected)*.
5. Keep this tab open while you make videos. The link stops when the Colab session ends (idle time-out
   or the session time limit); run all cells again and paste the new URL.

## 1. Check the GPU and memory

In [ ]:
import os
import subprocess

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch

assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > choose T4, L4 or A100 > Save'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
SM_MAJOR = torch.cuda.get_device_capability(0)[0]

# Profile for this GPU.
#  * T5_ON_GPU: the 11 GB UMT5 text encoder lives on the GPU (fast) instead of
#    the CPU (slow, and it needs ~17 GB of system RAM while loading).
#  * OFFLOAD: move models between CPU and GPU between stages (saves VRAM, costs time).
#  * MAX_STEPS: a T4 (no bfloat16 hardware) needs ~1 min per step for a 5 s clip;
#    capping the steps keeps a clip inside the studio's 45-minute wait.
T5_ON_GPU = VRAM_GB >= 20            # L4 (24 GB), A100 (40/80 GB), ...
OFFLOAD = VRAM_GB < 40               # A100 keeps everything resident
MAX_STEPS = 50 if SM_MAJOR >= 8 else 25
SAMPLE_SHIFT = 8.0                   # Wan2.1 README: 8-12 for the 1.3B model

print(f'GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GB | system RAM: {RAM_GB:.1f} GB')
print(f'Profile: text encoder on {"GPU" if T5_ON_GPU else "CPU"} | offload={OFFLOAD} | max steps={MAX_STEPS}')
if not T5_ON_GPU and RAM_GB < 20:
    raise RuntimeError(
        f'This runtime has only {RAM_GB:.0f} GB of system RAM. On a {GPU_NAME} the 11 GB text encoder '
        'runs on the CPU and loading needs about 17 GB, so the session would crash. Fix: '
        'Runtime > Change runtime type > turn ON "High-RAM" (Colab Pro), or choose an L4 / A100 GPU, '
        'then run all cells again.')
print('GPU check passed')

## 2. Install dependencies + fetch the Wan2.1 code

In [ ]:
import importlib

%cd /content
![ -d Wan2.1 ] || git clone --depth 1 https://github.com/Wan-Video/Wan2.1.git
%cd /content/Wan2.1
# Colab already ships torch, torchvision and NumPy 2. Wan's requirements pin
# numpy<2 (installing it breaks Colab's other packages with "compiled using
# NumPy 1.x" errors until a restart) and flash_attn (a 30+ minute compile that
# often fails). Both are skipped; attention uses PyTorch SDPA instead (cell 4).
!grep -viE '^(numpy|torch|torchvision|flash[_-]?attn|gradio)([<>=!~ ;\[]|$)' requirements.txt > /tmp/wan_reqs.txt
!pip install -q -r /tmp/wan_reqs.txt
# gradio 5/6 speaks the protocol of the studio's gradio_client 2.x.
!pip install -q "gradio>=5,<7" ftfy easydict imageio-ffmpeg
importlib.invalidate_caches()
for _pkg in ('easydict', 'ftfy', 'imageio_ffmpeg', 'gradio', 'diffusers', 'transformers'):
    importlib.import_module(_pkg)
import numpy

print('deps installed | gradio', importlib.import_module('gradio').__version__, '| numpy', numpy.__version__)

## 3. Download the T2V-1.3B weights (once per session)

In [ ]:
from huggingface_hub import snapshot_download

CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
snapshot_download('Wan-AI/Wan2.1-T2V-1.3B', local_dir=CKPT_DIR)
print('weights ready at', CKPT_DIR)

## 4. Load the model once
Kept in memory so every request is fast.

In [ ]:
import gc
import os
import sys
from contextlib import nullcontext

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, '/content/Wan2.1')

import torch
import wan
import wan.modules.attention as _attn
import wan.modules.model as _model
from wan.configs import WAN_CONFIGS

if not (_attn.FLASH_ATTN_2_AVAILABLE or _attn.FLASH_ATTN_3_AVAILABLE):
    try:
        from torch.nn.attention import SDPBackend
        from torch.nn.attention import sdpa_kernel as _sdpa_kernel

        def _fast_sdp_ctx():
            return _sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION])
    except Exception:
        def _fast_sdp_ctx():
            return nullcontext()

    def _sdpa_fallback(q, k, v, **kwargs):
        out_dtype = q.dtype
        if 'version' in kwargs:
            kwargs['fa_version'] = kwargs.pop('version')
        ql = kwargs.get('q_lens')
        if ql is not None and ql.numel() == 1 and int(ql[0]) == q.size(1):
            kwargs['q_lens'] = None
        if kwargs.get('q_lens') is None:
            kl = kwargs.get('k_lens')
            if kl is not None and kl.numel() == 1 and int(kl[0]) == k.size(1):
                kwargs['k_lens'] = None
        target_dtype = kwargs.get('dtype') or out_dtype
        # T4 (SM 7.5) Cutlass EFFICIENT_ATTENTION only runs on fp16/fp32, not bf16;
        # without casting here PyTorch silently uses MATH (~57.5 GiB OOM on 89f).
        if target_dtype == torch.bfloat16 and q.is_cuda and torch.cuda.get_device_capability(q.device)[0] < 8:
            kwargs['dtype'] = torch.float16
        else:
            kwargs.setdefault('dtype', out_dtype)

        large_dense = (
            q.is_cuda
            and kwargs.get('q_lens') is None
            and kwargs.get('k_lens') is None
            and q.size(1) * k.size(1) > 4_000_000
        )
        if large_dense:
            try:
                with _fast_sdp_ctx():
                    return _attn.attention(q, k, v, **kwargs).type(out_dtype)
            except RuntimeError:
                if not kwargs.get('causal', False) and kwargs.get('window_size', (-1, -1)) == (-1, -1) and q.size(1) > 1024:
                    chunks = [
                        _attn.attention(q[:, i:i + 1024], k, v, **kwargs).type(out_dtype)
                        for i in range(0, q.size(1), 1024)
                    ]
                    return torch.cat(chunks, dim=1)
                raise
        return _attn.attention(q, k, v, **kwargs).type(out_dtype)

    _attn.flash_attention = _sdpa_fallback
    _model.flash_attention = _sdpa_fallback
    print('flash-attn unavailable; using PyTorch SDPA fallback')


CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
cfg = WAN_CONFIGS['t2v-1.3B']

# Wan reads each checkpoint fully into RAM and then copies it into the model,
# which doubles peak memory (the text encoder alone is 11 GB). Memory-mapping
# the file lets the copy stream from disk instead.
_orig_torch_load = torch.load


def _mmap_torch_load(f, *args, **kwargs):
    if isinstance(f, (str, os.PathLike)) and 'mmap' not in kwargs:
        try:
            return _orig_torch_load(f, *args, mmap=True, **kwargs)
        except RuntimeError:
            pass            # not a zip-format checkpoint: load it normally
    return _orig_torch_load(f, *args, **kwargs)


print(f'Loading Wan2.1-T2V-1.3B on {GPU_NAME} (text encoder on {"GPU" if T5_ON_GPU else "CPU"})...')
torch.load = _mmap_torch_load
try:
    MODEL = wan.WanT2V(
        config=cfg, checkpoint_dir=CKPT_DIR, device_id=0,
        rank=0, t5_fsdp=False, dit_fsdp=False, use_usp=False,
        t5_cpu=not T5_ON_GPU,
    )
finally:
    torch.load = _orig_torch_load
gc.collect()

if OFFLOAD and not T5_ON_GPU:
    # Small GPU (T4): keep the VAE on the CPU until the decode step.
    MODEL.vae.model.to('cpu')
    _orig_vae_decode = MODEL.vae.decode

    def _offloaded_vae_decode(zs):
        try:
            MODEL.vae.model.to(MODEL.device)
            return _orig_vae_decode(zs)
        finally:
            MODEL.vae.model.to('cpu')
            gc.collect()
            torch.cuda.empty_cache()

    MODEL.vae.decode = _offloaded_vae_decode

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    # Same attention size as a 5.5 s clip (89 frames at 832x480).
    _q = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _k = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _v = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _kl = torch.tensor([35880], device='cuda', dtype=torch.int32)
    _out = _attn.flash_attention(_q, _k, _v, k_lens=_kl)
    assert _out.shape == _q.shape and _out.dtype == torch.bfloat16
    del _q, _k, _v, _kl, _out
gc.collect()
torch.cuda.empty_cache()
print('SDPA 35880-token test passed | peak VRAM MB:',
      round(torch.cuda.max_memory_allocated() / 1e6))
torch.cuda.reset_peak_memory_stats()
print('model loaded')

## 5. Start the worker server
Exposes `/generate` with the exact signature the local `ColabBackend` calls:
`(prompt, negative_prompt, width, height, num_frames, steps, guidance, seed)`.

In [ ]:
import gc
import hashlib
import os
import tempfile
import time

import gradio as gr
import torch
from wan.utils.utils import cache_video

# Wan2.1-T2V-1.3B is trained for these two sizes only.
SIZES = {'landscape': (832, 480), 'portrait': (480, 832)}
# Finished clips by request. A render cannot be stopped once it starts, so
# when the studio sends the same chunk again (after a restart or a retry) it
# waits in the queue behind the first render and then gets that clip at once
# instead of a second 20-minute render.
FINISHED = {}


def generate(prompt, negative_prompt, width, height, num_frames,
             steps, guidance, seed, progress=gr.Progress(track_tqdm=True)):  # noqa: B008 (Gradio's idiom)
    prompt = (prompt or '').strip()
    if not prompt:
        raise gr.Error('Empty prompt')
    width, height = SIZES['landscape'] if int(width) >= int(height) else SIZES['portrait']
    num_frames = max(5, min(int(num_frames), 129))
    if (num_frames - 1) % 4 != 0:            # Wan needs 4n+1 frames
        num_frames = (num_frames // 4) * 4 + 1
    steps = max(4, min(int(steps), MAX_STEPS))
    seed = int(seed)
    key = None
    if seed >= 0:
        key = hashlib.sha1(repr((prompt, negative_prompt or '', width, height, num_frames,
                                 steps, round(float(guidance), 3), seed)).encode()).hexdigest()
        done = FINISHED.get(key)
        if done and os.path.isfile(done):
            print(f'same request as before; returning the finished clip ({prompt[:40]})', flush=True)
            return done
    else:
        seed = torch.randint(0, 2**31 - 1, (1,)).item()
    print(f'gen {width}x{height} f{num_frames} s{steps} seed{seed}: {prompt[:60]}',
          flush=True)
    started = time.time()
    video = None
    out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    try:
        video = MODEL.generate(
            prompt,
            size=(width, height),
            frame_num=num_frames,
            shift=SAMPLE_SHIFT,
            sampling_steps=steps,
            guide_scale=float(guidance),
            n_prompt=negative_prompt or '',
            seed=seed,
            offload_model=OFFLOAD,
        )
        saved = cache_video(video[None], save_file=out, fps=16,
                            nrow=1, normalize=True, value_range=(-1, 1))
        if not saved or not os.path.isfile(out) or os.path.getsize(out) <= 0:
            raise RuntimeError(f'cache_video failed to write non-empty MP4 to {out}')
    except torch.cuda.OutOfMemoryError as exc:
        if os.path.exists(out):
            os.remove(out)
        raise gr.Error(f'CUDA out of memory on {GPU_NAME}: {str(exc)[:200]}') from exc
    except Exception:
        if os.path.exists(out):
            os.remove(out)
        raise
    finally:
        del video
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f'done {os.path.getsize(out)} bytes in {time.time() - started:.0f}s', flush=True)
    if key:
        FINISHED[key] = out
    return out


with gr.Blocks(title='Wan2.1 Colab Worker') as worker:
    gr.Markdown(f'### Wan2.1 T2V-1.3B worker on {GPU_NAME} — leave this tab open')
    with gr.Row():
        p  = gr.Textbox(label='prompt')
        np_ = gr.Textbox(label='negative_prompt')
    with gr.Row():
        w = gr.Number(832, label='width');  h = gr.Number(480, label='height')
        nf = gr.Number(81, label='num_frames'); st = gr.Number(30, label='steps')
        g = gr.Number(6.0, label='guidance'); sd = gr.Number(-1, label='seed')
    vid = gr.Video(label='result')
    gr.Button('Generate', variant='primary').click(
        generate, [p, np_, w, h, nf, st, g, sd], vid, api_name='generate')

worker.queue(max_size=20, default_concurrency_limit=1)
print('\n' + '='*70)
print('COPY THE  https://xxxxx.gradio.live  URL BELOW into VideoStudio Settings:')
print('='*70)
worker.launch(share=True, show_error=True)

## Troubleshooting

| What you see | What to do |
|---|---|
| Cell 1 stops with *only N GB of system RAM* | Runtime → Change runtime type → turn on **High-RAM**, or pick **L4 / A100**. |
| *Your session crashed after using all available RAM* | Same as above. |
| `CUDA out of memory` in the studio's log | Restart the runtime (Runtime → Restart session) and run all cells again. |
| Studio says *not responding* / *offline or expired* | The Colab session ended. Run all cells again and paste the **new** URL. |
| A clip takes very long on T4 | Normal: about 1 minute per step for 5 seconds of video. L4 / A100 are several times faster. |